# 01 — Data Preparation for SFT Intent Classifier

This notebook prepares **Veridian Systems** historical IT ticket data for fine-tuning
a lightweight intent classifier on Together.ai.
It converts raw tickets to the SFT JSONL format and splits the data into
train / val / test sets. The split files are read directly by `02_train_classifier.ipynb`.

In [ ]:
!uv add mistralai pandas scikit-learn python-dotenv rich

In [ ]:
import json
import os
import uuid
from pathlib import Path

import pandas as pd
from sklearn.model_selection import train_test_split
from rich.console import Console
from rich.table import Table
from rich.panel import Panel

console = Console()

# ── Paths ─────────────────────────────────────────────────────────────────
DATA_DIR        = Path("data")
RAW_FILE        = DATA_DIR / "raw_tickets.json"
SYNTHETIC_FILE  = DATA_DIR / "raw_tickets_synthetic.json"
TRAIN_FILE      = DATA_DIR / "train.jsonl"
VAL_FILE        = DATA_DIR / "val.jsonl"
TEST_FILE       = DATA_DIR / "test.jsonl"

# ── SFT classifier system prompt ──────────────────────────────────────────
CLASSIFIER_SYSTEM_PROMPT = (
    "Classify the IT support request into exactly one of the following categories: "
    "access_request, security_incident, hardware_issue, software_issue, onboarding, "
    "payments_incident, expense_request, general_question. "
    "Respond with only the category label, nothing else."
)

# ── All valid intent labels (order matters for reproducibility) ───────────
INTENT_LABELS = [
    "access_request",
    "security_incident",
    "hardware_issue",
    "software_issue",
    "onboarding",
    "payments_incident",
    "expense_request",
    "general_question",
]

RANDOM_STATE = 42

## 1. Overview

### What this notebook does

1. Loads `data/raw_tickets.json` — 60 labelled Veridian IT support tickets across
   8 intent categories.
2. Converts each ticket to the **SFT JSONL format**:
   ```json
   {"messages": [
     {"role": "system", "content": "<classifier system prompt>"},
     {"role": "user",   "content": "<ticket text>"},
     {"role": "assistant", "content": "<intent label>"}
   ]}
   ```
   The assistant turn contains only the intent label — the model learns to output
   exactly one of the 8 labels given a ticket description.
3. Splits the data **70 % train / 15 % val / 15 % test**, stratified by intent
   so every class is proportionally represented in all three splits.
4. Saves `data/train.jsonl`, `data/val.jsonl`, and `data/test.jsonl`.
   > **Do not regenerate these files after the classifier has been trained** —
   > changing the split would invalidate test-set evaluation results.
5. Uploads `train.jsonl` and `val.jsonl` to the Mistral Files API and
   prints the file IDs needed for `02_train_classifier.ipynb`.

### Intent categories

| Label | Description |
|---|---|
| `access_request` | Nexus, Prism, GitHub, AWS, Okta provisioning / offboarding |
| `security_incident` | SEV1/SEV2/SEV3 events — phishing, stolen device, suspicious login |
| `hardware_issue` | MacBook M3 MDM, monitors, docks, peripherals, battery |
| `software_issue` | Okta MFA, Slack, Zoom, IDE licences, Docker, Nexus auth |
| `onboarding` | Day-1 setup, Helix on-call, Prism access, badge, VPN |
| `payments_incident` | prod-payments degradation — always P1, paged via Helix |
| `expense_request` | Home office budget, ergonomics, L&D, Adobe, internet stipend |
| `general_question` | Policy queries, Jira routing, WiFi, HR redirects |

## 2. Load and explore raw tickets

In [ ]:
with open(RAW_FILE) as f:
    raw_tickets = json.load(f)

# Merge synthetic tickets if they have been generated
SYNTHETIC_FILE = DATA_DIR / "raw_tickets_synthetic.json"
if SYNTHETIC_FILE.exists():
    with open(SYNTHETIC_FILE) as f:
        synthetic_tickets = json.load(f)
    raw_tickets = raw_tickets + synthetic_tickets
    console.print(f"[dim]Merged {len(synthetic_tickets)} synthetic tickets from {SYNTHETIC_FILE.name}[/dim]")

df = pd.DataFrame(raw_tickets)
console.print(f"Loaded [bold]{len(df)}[/bold] tickets across [bold]{df['intent'].nunique()}[/bold] intent categories\n")

# Validate: every ticket must have exactly the fields we need
required_fields = {"id", "text", "intent", "priority"}
missing = [t["id"] for t in raw_tickets if not required_fields.issubset(t.keys())]
if missing:
    raise ValueError(f"Tickets missing required fields: {missing}")

# Validate: all intents are from the known set
unknown_intents = set(df["intent"]) - set(INTENT_LABELS)
if unknown_intents:
    raise ValueError(f"Unknown intent labels in data: {unknown_intents}")

console.print("[green]✓[/green] All fields present and intent labels valid\n")
df[["id", "intent", "priority", "text"]].head(8)

In [ ]:
dist = df["intent"].value_counts().reindex(INTENT_LABELS)

table = Table(title="Class Distribution — raw_tickets.json", show_header=True, header_style="bold")
table.add_column("Intent", style="cyan", min_width=22)
table.add_column("Count", justify="right", style="magenta")
table.add_column("Share", justify="right")
table.add_column("Bar", min_width=20)

for intent, count in dist.items():
    bar = "█" * count
    table.add_row(intent, str(count), f"{count / len(df):.0%}", bar)

console.print(table)
console.print(
    "[dim]Note: 6–8 examples per class is sufficient to demonstrate the pipeline "
    "end-to-end. For production, aim for ≥ 50 examples per class.[/dim]"
)

## 3. Generate synthetic tickets (optional — recommended)

60 examples (~7 per class) is enough to demonstrate the pipeline but too few
for reliable fine-tuning. This section generates **30 additional synthetic
tickets per class** (240 new examples) using `mistral-small-latest`, bringing
the total to ~300 examples (~37 per class) — enough for a meaningful F1 improvement.

The synthetic tickets are saved to `data/raw_tickets_synthetic.json` and merged
with the originals before splitting. `raw_tickets.json` is never modified.

Skip this cell if you want to use the original 60-example dataset only.

In [ ]:
try:
    from google.colab import userdata
    MISTRAL_API_KEY = userdata.get("MISTRAL_API_KEY")
except ImportError:
    from dotenv import load_dotenv
    load_dotenv(override=True)
    MISTRAL_API_KEY = os.getenv("MISTRAL_API_KEY")

from mistralai.client import Mistral

N_PER_CLASS = 30   # synthetic tickets to generate per intent class

_CLASS_DESCRIPTIONS = {
    "access_request":    "account provisioning, permission changes, or access removal — e.g. Nexus artifact repo, Prism data warehouse, GitHub org, AWS IAM, Okta group, offboarding",
    "security_incident": "suspected breach, phishing email, ransomware, stolen/lost device, suspicious login, unauthorised data access — any SEV-tier security event",
    "hardware_issue":    "broken or malfunctioning physical equipment — MacBook, monitor, dock, keyboard, mouse, webcam, headset, battery, charging cable, printer",
    "software_issue":    "software install requests, licence renewals, version upgrades, SSO/MFA problems — Okta Verify, Slack, Zoom, JetBrains, Docker, VS Code, Nexus auth",
    "onboarding":        "new-hire day-1 setup — MDM enrolment, Slack/Okta account creation, Helix on-call rotation, Prism read access, VPN profile, badge, laptop provisioning",
    "payments_incident": "any degradation or error touching the prod-payments service — webhook delays, 5xx errors, duplicate charges, Braintree/Stripe failures, P99 latency spikes",
    "expense_request":   "reimbursement or budget approval — home office equipment, ergonomic desk, L&D course, conference ticket, internet stipend, software subscription, doctor-recommended gear",
    "general_question":  "policy questions, how-to queries, SLA questions, Jira routing questions, WiFi help, HR redirects — requests that need an answer, not a ticket",
}

_GENERATION_PROMPT = """You are generating realistic IT support tickets for Veridian Systems, a B2B SaaS company (~800 employees).

Generate {n} distinct IT support ticket texts for the intent category: **{intent}**
Category description: {description}

Rules:
- Each ticket must be 1–4 sentences, written by an employee (first person, informal)
- Vary the writing style: some terse, some detailed, some urgent
- Use Veridian-specific internal names where natural: Nexus (artifact repo), Prism (data warehouse), Helix (on-call tool), prod-payments (critical payment service)
- Do NOT include the intent label in the ticket text
- Do NOT number the tickets or add any prefixes — output one ticket per line, nothing else
- Each ticket must be on a single line with no blank lines between them

Output exactly {n} lines."""

mistral_client = Mistral(api_key=MISTRAL_API_KEY)
synthetic_tickets: list[dict] = []

for intent, description in _CLASS_DESCRIPTIONS.items():
    console.print(f"Generating [bold]{N_PER_CLASS}[/bold] synthetic tickets for [cyan]{intent}[/cyan]…")

    response = mistral_client.chat.complete(
        model="mistral-small-latest",
        messages=[{
            "role": "user",
            "content": _GENERATION_PROMPT.format(
                n=N_PER_CLASS,
                intent=intent,
                description=description,
            ),
        }],
        temperature=0.9,
        max_tokens=2000,
    )

    raw_text = response.choices[0].message.content.strip()
    lines = [l.strip() for l in raw_text.splitlines() if l.strip()]

    if len(lines) != N_PER_CLASS:
        console.print(f"  [yellow]Warning: expected {N_PER_CLASS} lines, got {len(lines)} — using what we got[/yellow]")

    for line in lines:
        synthetic_tickets.append({
            "id":       f"SYN-{uuid.uuid4().hex[:6].upper()}",
            "text":     line,
            "intent":   intent,
            "priority": "P2",
        })

SYNTHETIC_FILE.write_text(json.dumps(synthetic_tickets, indent=2))
console.print(Panel(
    f"Generated [bold]{len(synthetic_tickets)}[/bold] synthetic tickets\n"
    f"Saved → [cyan]{SYNTHETIC_FILE}[/cyan]\n"
    f"[dim]Re-run cell-5 (load) to merge before splitting.[/dim]",
    title="[green]Synthetic data ready[/green]",
    border_style="green",
))

## 3. Convert to SFT JSONL format

Each training example is a JSON object with a `messages` list containing three turns:
a fixed system prompt, the ticket text as the user turn, and the intent label as the
assistant turn.

```json
{"messages": [
  {"role": "system",    "content": "<CLASSIFIER_SYSTEM_PROMPT>"},
  {"role": "user",      "content": "<ticket text>"},
  {"role": "assistant", "content": "<intent label>"}
]}
```

The assistant turn contains **only** the intent label — no punctuation, no explanation.
The fine-tuned model learns to reproduce this output given any new ticket.

At inference time `AdaptedAgent._classify()` sends the same system prompt and the
user's message; the model's response is parsed directly as the intent label.

In [ ]:
def ticket_to_example(ticket: dict) -> dict:
    """Convert a raw ticket dict to an SFT training example."""
    return {
        "messages": [
            {"role": "system",    "content": CLASSIFIER_SYSTEM_PROMPT},
            {"role": "user",      "content": ticket["text"]},
            {"role": "assistant", "content": ticket["intent"]},
        ]
    }


examples = [ticket_to_example(t) for t in raw_tickets]

console.print(f"Converted [bold]{len(examples)}[/bold] tickets to SFT format\n")

console.print("[bold]3 example records:[/bold]")
for ex in examples[:3]:
    msgs = ex["messages"]
    console.print(Panel(
        f"[bold cyan]intent:[/bold cyan] {msgs[-1]['content']}\n"
        f"[bold]text:[/bold] {msgs[1]['content']}",
        expand=False,
    ))

## 4. Train / validation / test split

Splits: **70 % train · 15 % val · 15 % test**, stratified by intent.

Stratification ensures every intent class appears in all three splits in roughly
the same proportion as in the full dataset — important when some classes
(e.g. `payments_incident`) have fewer examples than others.

The split is done in two passes:
1. Separate 70 % train from 30 % remainder (stratified).
2. Split the 30 % remainder 50/50 into val and test (stratified).

In [ ]:
all_labels = [ex["messages"][-1]["content"] for ex in examples]

# Pass 1: 70% train, 30% temp
train_examples, temp_examples = train_test_split(
    examples,
    test_size=0.30,
    random_state=RANDOM_STATE,
    stratify=all_labels,
)

# Pass 2: split temp 50/50 → val (15%) and test (15%)
temp_labels = [ex["messages"][-1]["content"] for ex in temp_examples]
val_examples, test_examples = train_test_split(
    temp_examples,
    test_size=0.50,
    random_state=RANDOM_STATE,
    stratify=temp_labels,
)

console.print(
    f"Split complete: "
    f"train=[bold]{len(train_examples)}[/bold]  "
    f"val=[bold]{len(val_examples)}[/bold]  "
    f"test=[bold]{len(test_examples)}[/bold]"
)

In [ ]:
def count_by_label(split: list[dict]) -> dict[str, int]:
    counts: dict[str, int] = {label: 0 for label in INTENT_LABELS}
    for ex in split:
        counts[ex["messages"][-1]["content"]] += 1
    return counts


train_counts = count_by_label(train_examples)
val_counts   = count_by_label(val_examples)
test_counts  = count_by_label(test_examples)

table = Table(
    title=f"Examples per split per category  (total {len(examples)})",
    show_header=True,
    header_style="bold",
)
table.add_column("Intent", style="cyan", min_width=22)
table.add_column("Train",  justify="right", style="green")
table.add_column("Val",    justify="right", style="yellow")
table.add_column("Test",   justify="right", style="magenta")
table.add_column("Total",  justify="right")

for label in INTENT_LABELS:
    tr, v, te = train_counts[label], val_counts[label], test_counts[label]
    table.add_row(label, str(tr), str(v), str(te), str(tr + v + te))

table.add_section()
table.add_row(
    "[bold]TOTAL[/bold]",
    f"[bold]{len(train_examples)}[/bold]",
    f"[bold]{len(val_examples)}[/bold]",
    f"[bold]{len(test_examples)}[/bold]",
    f"[bold]{len(examples)}[/bold]",
)

console.print(table)

## 5. Save splits

In [ ]:
def save_jsonl(split: list[dict], path: Path) -> None:
    """Write a list of dicts to a JSONL file, one record per line."""
    with open(path, "w") as f:
        for record in split:
            f.write(json.dumps(record) + "\n")
    console.print(f"  Saved [bold]{len(split):>3}[/bold] examples → [cyan]{path}[/cyan]")


console.print("[bold]Writing JSONL files:[/bold]")
save_jsonl(train_examples, TRAIN_FILE)
save_jsonl(val_examples,   VAL_FILE)
save_jsonl(test_examples,  TEST_FILE)

# ── Spot-check the first line of each file ────────────────────────────────
console.print("\n[bold]Format check (first record of each file):[/bold]")
for path in (TRAIN_FILE, VAL_FILE, TEST_FILE):
    with open(path) as f:
        first = json.loads(f.readline())
    msgs = first["messages"]
    console.print(
        f"  [cyan]{path.name}[/cyan]  "
        f"intent=[green]{msgs[-1]['content']}[/green]  "
        f"text=[dim]{msgs[1]['content'][:72]}…[/dim]"
    )

## 6. Forge context

### How this step relates to a production Forge deployment

In a production **Forge** deployment, the domain-adaptation step would not be
a supervised classification fine-tune on labelled tickets.
Instead it would be **continued pre-training on unlabelled enterprise text** —
ticket bodies, internal wiki pages, runbook documents, architecture decision
records, and Slack incident threads — allowing the base model to absorb
Veridian-specific terminology (Nexus, Prism, Helix, prod-payments, SEV tiers)
at the level of the full model weights.

**SFT fine-tuning is a targeted, supervised variant of the same fine-tuning
infrastructure.** Rather than adapting the entire model to a new text domain,
it fine-tunes the model weights to map input text to a discrete label — using
the same underlying training pipeline as Forge.

| | SFT fine-tuning (this demo) | Forge continued pre-training |
|---|---|---|
| **Input data** | Labelled `{messages}` examples | Unlabelled text corpus |
| **Training signal** | Cross-entropy on known labels | Next-token prediction |
| **What adapts** | Full model weights (small base) | Full model weights |
| **Output** | Intent classifier model ID | Domain-adapted base model |
| **Data residency** | La Plateforme (public) or Forge | Forge (on-prem / private cloud) |
| **Typical data size** | Tens to thousands of examples | Gigabytes of text |

The Python SDK code in this notebook and in `02_train_classifier.ipynb` would
require **only a `server_url` change** to target a Forge endpoint instead of
La Plateforme — the file upload, job creation, and polling logic is identical.